In [1]:
import pandas as pd
import os

In [2]:
base_path = r"C:\Users\hp\Desktop\customer-churn-analysis-python-\data\cleaned_data"

In [3]:
customers = pd.read_csv(os.path.join(base_path, "customers_cleaned.csv"))

In [4]:
transactions = pd.read_csv(os.path.join(base_path, "transactions_cleaned.csv"))

In [5]:
plans = pd.read_csv(os.path.join(base_path, "plans_cleaned.csv"))

In [6]:
support_tickets = pd.read_csv(os.path.join(base_path, "support_tickets_cleaned.csv"))

In [7]:
customer_usage = pd.read_csv(os.path.join(base_path, "customer_usage_cleaned.csv"))

In [8]:
print("customers:",customers.shape)
print("plans:", plans.shape)
print("transactions:", transactions.shape)
print("support_tickets:", support_tickets.shape)
print("customer_usage:", customer_usage.shape)

customers: (15000, 12)
plans: (4, 3)
transactions: (80000, 6)
support_tickets: (20000, 6)
customer_usage: (45000, 6)


In [9]:
transactions["customer_id"].isin(customers["customer_id"])

0        True
1        True
2        True
3        True
4        True
         ... 
79995    True
79996    True
79997    True
79998    True
79999    True
Name: customer_id, Length: 80000, dtype: bool

In [10]:
(~support_tickets["customer_id"].isin(customers["customer_id"])).sum()

np.int64(0)

In [11]:
(~customer_usage["customer_id"].isin(customers["customer_id"])).sum()

np.int64(0)

In [12]:
customer_plan = customers.merge(plans, on="plan_id", how="left")

In [13]:
customer_plan.shape

(15000, 14)

In [14]:
print("Before merge:",len(customers))
print("After merge:",len(customer_plan))

Before merge: 15000
After merge: 15000


In [15]:
customer_plan.isnull().sum()

customer_id         0
customer_name       0
gender              0
age                 0
city                0
state               0
join_date           0
plan_id             0
contract_type       0
internet_service    0
tenure_months       0
churn               0
plan_name           0
monthly_fee         0
dtype: int64

In [16]:
transaction_summary = transactions.groupby("customer_id").agg(
    total_amount = ("amount", "sum"),
    avg_transaction_amount = ("amount", "mean"),
    transaction_count = ("transaction_id","count")
).reset_index()

In [17]:
transaction_summary.head()

,customer_id,total_amount,avg_transaction_amount,transaction_count
0,1,10237,1462.428571,7
1,2,4012,1003.000000,4
2,3,2397,2397.000000,1
3,4,6286,1571.500000,4
4,5,4723,1180.750000,4


In [18]:
transaction_summary.shape

(14927, 4)

In [19]:
transaction_summary["customer_id"].nunique()

14927

In [20]:
transactions["payment_status"].value_counts()

payment_status
Paid      65691
Late       9501
Failed     4808
Name: count, dtype: int64

In [21]:
transactions["failed_payment"] = (
    transactions["payment_status"]=="Failed"
    ).astype(int)

transactions["late_payment"] = (
    transactions["payment_status"]=="Late"
).astype(int)

In [22]:
transactions[[
    "payment_status",
    "failed_payment",
    "late_payment"
]].head(10)

,payment_status,failed_payment,late_payment
0,Paid,0,0
1,Paid,0,0
2,Late,0,1
3,Paid,0,0
4,Paid,0,0
5,Paid,0,0
6,Paid,0,0
7,Paid,0,0
8,Paid,0,0
9,Late,0,1


In [23]:
payment_summary = transactions.groupby("customer_id").agg(
    failed_payment_count=("failed_payment", "sum"),
    late_payment_count=("late_payment", "sum")
).reset_index()

In [24]:
payment_summary.head()

,customer_id,failed_payment_count,late_payment_count
0,1,0,0
1,2,0,0
2,3,0,0
3,4,0,1
4,5,0,0


In [25]:
payment_summary.shape

(14927, 3)

In [26]:
payment_summary["customer_id"].nunique()

14927

In [27]:
transaction_summary = transaction_summary.merge(
    payment_summary,
    on = "customer_id",
    how = "left"
)

In [28]:
transaction_summary.head()

,customer_id,total_amount,avg_transaction_amount,transaction_count,failed_payment_count,late_payment_count
0,1,10237,1462.428571,7,0,0
1,2,4012,1003.000000,4,0,0
2,3,2397,2397.000000,1,0,0
3,4,6286,1571.500000,4,0,1
4,5,4723,1180.750000,4,0,0


In [29]:
support_tickets["issue_type"].value_counts()

issue_type
Billing      3404
Refund       3356
Technical    3352
Speed        3336
Login        3305
Network      3247
Name: count, dtype: int64

In [30]:
support_tickets["priority"].value_counts()

priority
Medium    6764
High      6648
Low       6588
Name: count, dtype: int64

In [31]:
support_tickets["high_priority"]=(
    support_tickets["priority"]=="high"
).astype(int)

In [32]:
support_summary = support_tickets.groupby("customer_id").agg(
    support_ticket_count = ("ticket_id", "count"),
    avg_resolution_days = ("resolution_days", "mean"),
    avg_satisfaction_score = ("satisfaction_score", "mean"),
    high_priotity_ticket_count = ("high_priority","sum")
).reset_index()

In [33]:
support_summary.head()

,customer_id,support_ticket_count,avg_resolution_days,avg_satisfaction_score,high_priotity_ticket_count
0,1,2,10.0,4.0,0
1,2,1,5.0,4.0,0
2,3,2,4.5,4.5,0
3,4,2,15.5,4.0,0
4,5,2,9.0,2.0,0


In [34]:
support_summary.shape

(11132, 5)

In [35]:
support_summary["customer_id"].nunique()

11132

In [37]:
customer_usage.head()

,usage_id,customer_id,month,data_usage_gb,call_minutes,sms_count
0,1,11659,2024-05-17,370.63,3655,117
1,2,9850,2024-11-17,303.87,1629,325
2,3,2878,2024-05-13,261.17,3046,182
3,4,12507,2024-03-19,188.82,462,433
4,5,11516,2024-02-04,218.29,3818,287


In [38]:
customer_usage["customer_id"].nunique()

14240

In [39]:
customer_usage.groupby("customer_id").size().describe()

count    14240.000000
mean         3.160112
std          1.637165
min          1.000000
25%          2.000000
50%          3.000000
75%          4.000000
max         12.000000
dtype: float64

In [40]:
usage_summary = customer_usage.groupby("customer_id").agg(
    avg_data_usage_gb = ("data_usage_gb", "mean"),
    avg_call_minutes = ("call_minutes", "mean"),
    avg_sms_count = ("sms_count", "mean"),
    max_data_usage_gb = ("data_usage_gb", "max")
).reset_index()

In [41]:
usage_summary.head()

,customer_id,avg_data_usage_gb,avg_call_minutes,avg_sms_count,max_data_usage_gb
0,1,248.916667,910.00,186.00,411.33
1,2,36.940000,1542.00,67.00,36.94
2,3,231.285000,2144.75,388.25,438.66
3,4,416.520000,1637.00,326.50,470.11
4,5,306.990000,3170.00,281.60,466.48


In [42]:
usage_summary.shape

(14240, 5)

In [43]:
usage_summary["customer_id"].nunique()

14240

In [ ]:
# Combine Transaction Features se start karna hai